# Fruit Classifier (Weight & Sweetness) - Logits + Softmax Demo
A small neural network that classifies a fruit as Apple (0), Banana (1), or Orange (2)
based on weight (g) and sweetness rating. Used mainly to demonstrate the difference
between raw logits and Softmax probabilities.

In [1]:
import tensorflow as tf
import numpy as np

tf.random.set_seed(42)  # for reproducible results

class_names = ["Apple", "Banana", "Orange"]

# Features: [weight (g), sweetness rating]
X_train = np.array([
    [150, 7],    # Apple
    [120, 9],    # Banana
    [180, 4],    # Orange
    [140, 8],    # Apple
    [110, 9.5],  # Banana
], dtype=np.float32)

y_train = np.array([0, 1, 2, 0, 1])  # Apple, Banana, Orange, Apple, Banana


In [2]:
# Build the model without a Softmax activation on the last layer,
# so it outputs raw logits directly (see the softmax step below)
model = tf.keras.Sequential([
    tf.keras.Input(shape=(2,)),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dense(3)  # 3 logits, one per class
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history = model.fit(X_train, y_train, epochs=200, verbose=0)
print(f"Final training loss: {history.history['loss'][-1]:.4f}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.4f}")


Final training loss: 2.1804
Final training accuracy: 0.6000


## Predicting a new sample

In [3]:
# Example: weight 115g, sweetness 9
sample = np.array([[115, 9]], dtype=np.float32)

# 1. Get the raw logits from the model
logits = model.predict(sample, verbose=0)

# 2. Apply Softmax explicitly to convert logits (numbers that are hard to
#    interpret directly) into probabilities that sum to 1.
# Softmax is used for multi-class classification because it turns model
# outputs into a probability distribution over all classes - unlike
# Sigmoid, which is more appropriate for binary classification.
probabilities = tf.nn.softmax(logits).numpy()

# 3. Pick the class with the highest probability
# axis=-1 keeps this correct even if `sample` contains multiple rows
predicted_class = np.argmax(probabilities, axis=-1)[0]

print("Probabilities:", probabilities)
print(f"Predicted Class: {predicted_class} ({class_names[predicted_class]})")


Probabilities: [[0.15564664 0.6860042  0.15834911]]
Predicted Class: 1 (Banana)
